# 02 — The Value object
Lecture 1, ~14-18% (search: "class value takes a single scalar"). Phase 1, item 2 in [PLAN](../knowledge/PLAN.md). Note: [02-value-object](../knowledge/notes/02-value-object.md). Prereq: [python-dunder-methods](../knowledge/prereqs/python-dunder-methods.md).

**Where this fits.** In notebook 01, every slope cost one full re-run of the expression. To beat that, the expression has to remember how it was built. Today: a number that keeps a receipt.

**Done when** `d = a*b + c` gives `Value(data=4.0)`, and from `d` alone you can find out what operation made it and which values went in.

Pure Python. No numpy, no torch.

## 1. Wrap a number
A class `Value` that takes one number and stores it as `.data`. Give it a `__repr__` so `Value(2.0)` displays as `Value(data=2.0)`.

Before adding `__repr__`, look at what the notebook shows for a `Value`. Then add it.

In [ ]:
class Value:
    def __init__(self, data):
        self.data = data
        # this field will be used to track the previous Value in the computational graph
        self._prev = ()
        # this field will be used to track the operation that produced this Value
        self._op = ""

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        # create a new Value object to represent the result of the addition
        out = Value(self.data + other.data)
        # set the _prev field to track the previous Values in the computational graph
        out._prev = (self, other)
        # set the _op field to indicate that this Value was produced by addition
        out._op = "+"
        return out

    def __mul__(self, other):
        # create a new Value object to represent the result of the multiplication
        out = Value(self.data * other.data)
        # set the _prev field to track the previous Values in the computational graph
        out._prev = (self, other)
        # set the _op field to indicate that this Value was produced by multiplication
        out._op = "*"
        return out

## 2. Add two Values
`a = Value(2.0)`, `b = Value(-3.0)`. Try `a + b` **before** writing anything, and read the error.

Then add `__add__`. It returns a **new** Value. Expect `Value(data=-1.0)`.

Question: inside `__add__`, is `self.data + other.data` a Value plus a Value, or a float plus a float?

In [ ]:
# a = Value(2.0)
# b = Value(-3.0)
# print(a + b)  # Unsupported operand type(s) for +: 'Value' and 'Value'

a = Value(2.0)
b = Value(-3.0)
print(a + b)  # Value(data=-1.0)

## 3. Multiply, and rebuild d
Add `__mul__`. Then `c = Value(10.0)` and `d = a*b + c`. Expect `Value(data=4.0)`, the same `d` as notebook 01 section 5.

Also call the dunders by hand, e.g. `a.__mul__(b).__add__(c)`, and check you get the same thing.

In [ ]:
c = Value(10.0)
d = a * b + c
print(d)  # Value(data=4.0)
print(a.__mul__(b).__add__(c))  # Value(data=4.0), same thing by hand

## 4. Keep the receipt: children
Give `Value` a way to remember which Values made it, stored in `self._prev`.

**What you built:** `_prev` starts as an empty tuple `()`. Inside `__add__` / `__mul__`, after creating `out`, you set `out._prev = (self, other)`.
**Karpathy's version:** passes a `_children` tuple into the constructor and stores it as a `set`. Same idea. The tuple-vs-set difference shows up in section 6, Q3.

Now `d._prev` should contain the -6.0 and the 10.0. What should `a._prev` be?

In [ ]:
print(a._prev) # ()
print(b._prev) # ()
print(c._prev) # ()
print(d._prev) # (Value(data=-6.0), Value(data=10.0))

## 5. Keep the receipt: the operation
Add `_op`: an empty string for values you create by hand, `'+'` or `'*'` for values made by an operation.

Then, starting from `d` only, print the whole recipe: d's op and children, then the op and children of the -6.0. Don't use `a`, `b` or `c` by name.

In [ ]:
def traverse_graph(value):
    print(value._op)
    for node in value._prev:
        print(node)
        traverse_graph(node)

# d = a * b + c
traverse_graph(d)

## 6. Break it
Predict each one, then run it.

1. `a + 1`. What happens, and why? (Look at which line fails.)
2. `1 + a`. Same error or different? Which object's dunder does Python try first?
3. `e = a * a`. How many children does `e._prev` have? Should it be 1 or 2? (Hint: `_prev` is a set.) Keep this in mind for item 6.
4. `x = Value(2.0); x.data = 5.0` after building `y = x * b`. Does `y.data` change? What does that tell you about when the math happens?

In [ ]:
# 1. a + 1
# print(a + 1)  # AttributeError: 'int' object has no attribute 'data'
# Answer: a.__add__(1)
# which line of __add__ fails, and what is `other` at that moment?
# this line fails: out = Value(self.data * other.data) and the `other` is 1 at that time

# 2. 1 + a
# print(1 + a)  # TypeError: unsupported operand type(s) for +: 'int' and 'Value'  (a different error)
# Answer: Python will call 1's __add__
# int gives up. What method does Python try on `a` next, before raising?
# Once both __add__ and __radd__ fail 
# (either by being missing or returning NotImplemented), 
# Python concludes the operation is impossible and raises:
# Python checks the left operand and ask if knows how to add Value type
# if left operand don't know it will return NotImplementedError and then
# Python asks the right operand and see if it can add `int`


# 3. a * a: one entry in the receipt or two?
e = a * a
print(e._prev)  # (Value(data=2.0), Value(data=2.0))
# Answer: for this instance, there will be two a's
# Keep for item 6: `a` feeds `e` along two paths.

# 4. x = Value(2.0); y = x * b; x.data = 5.0  → y.data stays -6.0
# Answer: the math happens once, when y is created.
# Follow-up: after the update step changes .data, what must happen before the next backward pass?
# Answer: a forward pass to calculate new loss using current weights

## Notes / findings
Your own words: what does `Value` store, and why can't a plain float do the same job?

**My answer:** Basically, a non-Value object like int, float once computed with some mathamatical expression like d = a * b + c when the computation is over and d is written there's no way for us to find out how does d was actually constructed all that info is lost now. That's where Value class comes into play. It keeps track of all the local operation and these local operation can be discovered from the final loss using d._prev